In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# 1.system tools and Ollama binary
!apt-get update && apt-get install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

# 2.Python dependencies
!pip install -q ollama rank_bm25 sentence-transformers datasets faiss-cpu pandas tqdm

import subprocess
import time
import os
import pandas as pd
import numpy as np
import faiss
import json
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
from tqdm.auto import tqdm

#ollama server
subprocess.Popen("ollama serve", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(15) # Wait for startup

import ollama
print("Pulling qwen3-embedding:0.6b...")
ollama.pull('qwen3-embedding:0.6b')

print("Loading Re-ranker (CPU mode)...")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device='cpu')
print("Setup Complete!")

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:4 https://cli.github.com/packages stable InRelease                         
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease               
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease     
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease   
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                   
Hit:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Setup Complete!


In [ ]:
from tqdm.auto import tqdm
import numpy as np
import faiss

# 1DevRev Dataset
dataset = load_dataset("devrev/search", "knowledge_base")
split_name = list(dataset.keys())[0]
kb_df = pd.DataFrame(dataset[split_name])
doc_texts = kb_df['text'].tolist()
doc_ids = kb_df['id'].tolist()
doc_titles = kb_df['title'].tolist()

# BM25 (Keyword) Index
print("Building BM25 index...")
tokenized_kb = [text.lower().split() for text in doc_texts]
bm25 = BM25Okapi(tokenized_kb)

#Embeddings using Ollama
def get_embeddings(texts, batch_size=32):
    all_embeddings = []
    pbar = tqdm(total=len(texts), desc="Ollama Embedding Progress")
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        for text in batch:
            try:
                res = ollama.embeddings(model="qwen3-embedding:0.6b", prompt=str(text)[:2000])
                all_embeddings.append(res["embedding"])
            except Exception as e:
                all_embeddings.append([0.0] * 1024) 
            pbar.update(1)
            
    pbar.close()
    return np.array(all_embeddings).astype('float32')

print(f"Generating embeddings for {len(doc_texts)} documents...")
embeddings_np = get_embeddings(doc_texts)

# FAISS Index 
print("Building FAISS Index...")
d = embeddings_np.shape[1] 
# using IndexFlatL2 
index = faiss.IndexFlatL2(d) 
index.add(embeddings_np)

print(f"Indexing complete! {index.ntotal} vectors ready in CPU index.")

Building BM25 index...
Generating embeddings for 65224 documents...


Ollama Embedding Progress:   0%|          | 0/65224 [00:00<?, ?it/s]

Building FAISS Index...
Indexing complete! 65224 vectors ready in CPU index.


In [ ]:
import subprocess
import time
import ollama

subprocess.Popen("ollama serve", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Waking up Ollama...")
time.sleep(10)

try:
    ollama.list()
    print("Ollama is back online!")
except:
    print("Still connecting... try running this cell once more.")

Waking up Ollama...
Ollama is back online!


In [ ]:
def winning_search(query, k_initial=50, k_final=10):
    """
     Pipeline: 
    1. Retrieval (Vector + BM25) 
    2. Fusion (RRF) 
    3. Re-ranking (Cross-Encoder)
    """
    #using same get_embeddings  
    query_vec = np.array([ollama.embeddings(model="qwen3-embedding:0.6b", prompt=str(query)[:2000])["embedding"]]).astype('float32')
    _, v_indices = index.search(query_vec, k_initial)
    vector_results = v_indices[0]
    
    # --- Step B: Keyword Search (BM25) ---
    t_query = str(query).lower().split()
    b_scores = bm25.get_scores(t_query)
    bm25_results = np.argsort(b_scores)[::-1][:k_initial]
    
    # Reciprocal Rank Fusion (RRF) ---

    fused_scores = {}
    k_rrf = 60 
    
    for rank, idx in enumerate(vector_results):
        fused_scores[idx] = fused_scores.get(idx, 0) + 1 / (k_rrf + rank)
        
    for rank, idx in enumerate(bm25_results):
        fused_scores[idx] = fused_scores.get(idx, 0) + 1 / (k_rrf + rank)
    
    # Sort by fused score 
    top_candidates = sorted(fused_scores.keys(), key=lambda x: fused_scores[x], reverse=True)[:20]
    
    #  Cross-Encoder Re-Ranking ---
    
    pairs = [[query, doc_texts[i]] for i in top_candidates]
    
    rr_scores = reranker.predict(pairs)
    
    # Sort candidates based on the Re-ranker's scores
    final_ranked_indices = [idx for _, idx in sorted(zip(rr_scores, top_candidates), reverse=True)]
    
  
    return [{
        "id": doc_ids[i],
        "text": doc_texts[i],
        "title": doc_titles[i]
    } for i in final_ranked_indices[:k_final]]

print("Testing search pipeline...")
test_q = "How do I integrate Bitbucket with DevRev?"
results = winning_search(test_q)
print(f"Top Result: {results[0]['title']}")

Testing search pipeline...
Top Result: Bitbucket | Integrate | Snap-ins | DevRev


In [ ]:
# 1. Load Test Queries
test_dataset = load_dataset("devrev/search", "test_queries")

print(f"Available splits in test_queries: {test_dataset.keys()}")
test_split = "test" if "test" in test_dataset.keys() else list(test_dataset.keys())[0]
test_data = test_dataset[test_split]

submission = []

print(f"Generating submission for {len(test_data)} queries...")
for item in tqdm(test_data, desc="Evaluating"):
    q_text = item.get('query', item.get('question', ""))
    q_id = item.get('query_id', item.get('id', "unknown"))

    results = winning_search(q_text)
    
    submission.append({
        "query_id": q_id,
        "query": q_text,
        "retrievals": results
    })


with open("test_queries_results.json", "w") as f:
    json.dump(submission, f, indent=2)

print(f"SUCCESS: Saved {len(submission)} results to 'test_queries_results.json'")

Available splits in test_queries: dict_keys(['test'])
Generating submission for 92 queries...


Evaluating:   0%|          | 0/92 [00:00<?, ?it/s]

In [ ]:
def run_local_evaluation(search_func, eval_data, limit=50):
    """
    Evaluates the search function against annotated ground truth.
    """
    total_recall = 0
    total_mrr = 0
    count = 0
    
    # We use a subset (e.g., first 50) to save time
    eval_subset = eval_data.select(range(min(limit, len(eval_data))))
    
    print(f"Evaluating {len(eval_subset)} queries...")
    
    for item in tqdm(eval_subset, desc="Evaluating Metrics"):
        query = item['query']
        golden_ids = [r['id'] for r in item['retrievals']]
        
        #  pipeline
        results = search_func(query)
        retrieved_ids = [r['id'] for r in results]
        found = any(gid in retrieved_ids for gid in golden_ids)
        if found:
            total_recall += 1
            
        # MRR (Mean Reciprocal Rank)
        for rank, rid in enumerate(retrieved_ids):
            if rid in golden_ids:
                total_mrr += 1 / (rank + 1)
                break
        
        count += 1
        
    print(f"\n--- LOCAL EVALUATION RESULTS (n={count}) ---")
    print(f"Recall@10: {total_recall/count:.4f} (Target: > 0.85)")
    print(f"MRR:       {total_mrr/count:.4f} (Target: > 0.70)")

annotated = load_dataset("devrev/search", "annotated_queries")["train"]
run_local_evaluation(winning_search, annotated)

annotated_queries.parquet:   0%|          | 0.00/448k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/291 [00:00<?, ? examples/s]

Evaluating 50 queries...


Evaluating Metrics:   0%|          | 0/50 [00:00<?, ?it/s]


--- LOCAL EVALUATION RESULTS (n=50) ---
Recall@10: 0.4800 (Target: > 0.85)
MRR:       0.2652 (Target: > 0.70)


In [22]:
import json

def verify_submission(file_path):
    with open(file_path, 'r') as f:
        data = json.load(f)
    
    if not isinstance(data, list):
        return "FAILED: Top level must be a list."
    
    sample = data[0]
    required_keys = ['query_id', 'query', 'retrievals']
    if not all(k in sample for k in required_keys):
        return f"FAILED: Missing keys. Found: {list(sample.keys())}"
    
    if len(sample['retrievals']) != 10:
        return f"WARNING: Found {len(sample['retrievals'])} results instead of 10. Check k_final."
    
    ret_sample = sample['retrievals'][0]
    if not all(k in ret_sample for k in ['id', 'text', 'title']):
        return "FAILED: Retrieval items must have 'id', 'text', and 'title'."
    
    return "✅ PASSED: Format is valid for DevRev submission."

print(verify_submission('test_queries_results.json'))

✅ PASSED: Format is valid for DevRev submission.
